# publish_risk_events_build
`gold_vuln_daily`의 어제↔오늘 비교(또는 BACKFILL 시 히스토리 전체)로 `EPSS_SPIKE`/`KEV_NEW` 판정 → `publish_risk_events`(append-only, `signal_id` UNIQUE).

- **KEV_NEW**: `silver_kev.valid_from == 오늘`인 CVE만 — `gold_vuln_daily.kev_listed`를 그대로 쓰면 등재된 다음날부터 계속 True라서 매일 재발화하므로, "오늘 새로 열린 버전"이라는 진짜 신호원을 따로 조회한다.
- **EPSS_SPIKE**: `is_systemic_day=true`인 날은 억제한다. `is_systemic_day`가 `NULL`(미판정)이면 억제하지 않되 `evidence`에 불확실성을 명시한다(시나리오⑨, 03_Gold_Schema.md §3.7 3상태 원칙).
- **주의**: 문서에 EPSS_SPIKE의 정확한 수치 임계값이 명시되어 있지 않아, 아래 `EPSS_SPIKE_ABS_DELTA_THRESHOLD`는 운영 중 조정이 필요한 잠정값이다.

In [ ]:
%run 00_gold_common

In [ ]:
EPSS_SPIKE_ABS_DELTA_THRESHOLD = 0.10  # 잠정값 -- 설계서에 정확한 수치 없음, DS팀과 협의 후 조정 필요

if run_mode in ("DAILY", "EVENT"):
    is_backfilled = False
    candidates = spark.table(TBL["gold_vuln_daily"]).filter(F.col("date_key") == snapshot_date)
    if run_mode == "EVENT":
        if not target_cve_ids:
            raise ValueError("EVENT 모드는 target_cve_ids가 필요합니다.")
        candidates = candidates.filter(F.col("cve_id").isin(target_cve_ids))
elif run_mode == "BACKFILL":
    if not target_cve_ids:
        raise ValueError("BACKFILL 모드는 target_cve_ids가 필요합니다.")
    is_backfilled = True
    candidates = spark.table(TBL["gold_vuln_daily"]).filter(F.col("cve_id").isin(target_cve_ids))
else:
    raise ValueError(f"Unsupported run_mode for publish_risk_events_build: {run_mode}")

candidates = candidates.select(
    "cve_id", "date_key", "epss_score", "epss_delta_1d", "kev_listed", "is_systemic_day"
)


In [ ]:
epss_spike = (
    candidates
    .filter(
        F.col("epss_delta_1d").isNotNull()
        & (F.abs(F.col("epss_delta_1d")) >= EPSS_SPIKE_ABS_DELTA_THRESHOLD)
        & (F.coalesce(F.col("is_systemic_day"), F.lit(False)) == False)
    )
    .withColumn("signal_type", F.lit("EPSS_SPIKE"))
    .withColumn("previous_value", F.col("epss_score") - F.col("epss_delta_1d"))
    .withColumn("new_value", F.col("epss_score"))
    .withColumn(
        "evidence",
        F.when(
            F.col("is_systemic_day").isNull(),
            F.lit("is_systemic_day=UNKNOWN (silver_epss_distribution 미생성) - 억제하지 않되 불확실성 있음"),
        ),
    )
)


In [ ]:
newly_listed_cves = (
    spark.table(TBL["kev"]).filter(F.col("valid_from") == snapshot_date).select("cve_id").distinct()
    if table_exists(TBL["kev"]) else spark.createDataFrame([], "cve_id string")
)

kev_new = (
    candidates.join(newly_listed_cves, on="cve_id", how="inner")
    .withColumn("signal_type", F.lit("KEV_NEW"))
    .withColumn("previous_value", F.lit(None).cast(DoubleType()))
    .withColumn("new_value", F.lit(1.0))
    .withColumn("evidence", F.lit(None).cast(StringType()))
)


In [ ]:
COMMON_COLS = ["cve_id", "date_key", "signal_type", "previous_value", "new_value",
                "is_systemic_day", "evidence"]

events_df = (
    epss_spike.select(*COMMON_COLS)
    .unionByName(kev_new.select(*COMMON_COLS))
    .withColumnRenamed("date_key", "event_date")
    .withColumn("detected_at", F.lit(INGESTED_AT))
    .withColumn("is_backfilled", F.lit(is_backfilled))
    .withColumn("batch_id", F.lit(batch_id))
)
events_df = events_df.withColumn(
    "signal_id", content_hash([F.col("cve_id"), F.col("event_date"), F.col("signal_type")], length=24)
)


In [ ]:
if table_exists(TBL["publish_risk_events"]):
    existing_ids = spark.table(TBL["publish_risk_events"]).select("signal_id")
    new_events = events_df.join(existing_ids, on="signal_id", how="left_anti")
else:
    new_events = events_df

new_count = new_events.count()
if new_count > 0:
    write_append(new_events, TBL["publish_risk_events"])

log_batch_run("publish_risk_events_build", "SUCCESS", record_count=new_count)
log_dq_metric("publish_risk_events_build", "new_signal_count", new_count, passed=True,
              detail=f"run_mode={run_mode}, is_backfilled={is_backfilled}")
print(f"[publish_risk_events_build] mode={run_mode} new_signals={new_count} batch_id={batch_id}")
